# Bootcamp Databricks - Dia 4

Notebook de apoio: geração de dados de amostra e testes de ingestão

Objetivo: gerar arquivos fictícios em um volume governado e testar quatro cenários do laboratório:

1. Full load didático
2. COPY INTO incremental e idempotente
3. Auto Loader com checkpoint e schema location
4. CDC didático preservado em Bronze

###  1. Preparando Ambiente

In [0]:
import json
from datetime import datetime, timedelta
from pyspark.sql.functions import current_timestamp, col

In [0]:

catalog_name = "capgemini_academy"

landing_schema = "landing"
bronze_schema = "bronze"
landing_volume = "files"
metadata_volume = "metadata"

source_folder = "pedidos"
checkpoint_folder = "_checkpoint/pedidos_autoloader"
schema_folder = "_schema/pedidos_autoloader"

table_full = f"{catalog_name}.{bronze_schema}.pedidos_full_load"
table_copy = f"{catalog_name}.{bronze_schema}.pedidos_copy_into"
table_auto = f"{catalog_name}.{bronze_schema}.pedidos_autoloader"
table_cdc_raw = f"{catalog_name}.{bronze_schema}.pedidos_cdc_raw"

source_path = f"/Volumes/{catalog_name}/{landing_schema}/{landing_volume}/{source_folder}"
metadata_path = f"/Volumes/{catalog_name}/{landing_schema}/{metadata_volume}"
checkpoint_path = f"{metadata_path}/{checkpoint_folder}"
schema_path = f"{metadata_path}/{schema_folder}"

print("Source path:     ", source_path)
print("Metadata path:   ", metadata_path)
print("Checkpoint path: ", checkpoint_path)
print("Schema path:     ", schema_path)

In [0]:
spark.sql(f"USE CATALOG {catalog_name}")
spark.sql(f"CREATE SCHEMA IF NOT EXISTS {catalog_name}.{landing_schema}")
spark.sql(f"CREATE SCHEMA IF NOT EXISTS {catalog_name}.{bronze_schema}")
spark.sql(f"CREATE VOLUME IF NOT EXISTS {catalog_name}.{landing_schema}.{landing_volume}")
spark.sql(f"CREATE VOLUME IF NOT EXISTS {catalog_name}.{landing_schema}.{metadata_volume}")

dbutils.fs.mkdirs(source_path)
dbutils.fs.mkdirs(checkpoint_path)
dbutils.fs.mkdirs(schema_path)

print("Ambiente preparado.")

### 2. Funções auxiliares
Os arquivos gerados são fictícios, pequenos e adequados para treinamento.

In [0]:
def write_jsonl(path, rows, overwrite=True):
    content = "\n".join(json.dumps(r, ensure_ascii=False) for r in rows)
    dbutils.fs.put(path, content, overwrite=overwrite)
    print(f"Arquivo gravado: {path} | linhas: {len(rows)}")

def list_landing(path=source_path):
    files = dbutils.fs.ls(path)
    display(files)
    return files

def make_order(order_id, customer_id, product_id, quantity, amount, minutes_offset, channel=None, status="novo"):
    base_time = datetime(2026, 9, 17, 14, 0, 0)
    row = {
        "pedido_id": str(order_id),
        "cliente_id": customer_id,
        "produto_id": product_id,
        "quantidade": int(quantity),
        "valor_total": float(amount),
        "data_pedido": (base_time + timedelta(minutes=minutes_offset)).isoformat(),
        "status": status
    }
    if channel is not None:
        row["canal_venda"] = channel
    return row

### 3. Gerar lote inicial
Este lote testa full load, COPY INTO e Auto Loader.

In [0]:

orders_batch_001 = [
    make_order(1001, "C001", "P100", 2, 59.90, 15, status="novo"),
    make_order(1002, "C002", "P200", 1, 89.50, 16, status="novo"),
    make_order(1003, "C003", "P300", 3, 149.70, 18, status="novo"),
    make_order(1004, "C001", "P400", 1, 39.90, 20, status="novo"),
    make_order('null', "C003", "P600", -5, 99.90, 20, status="novo"),
]

write_jsonl(f"{source_path}/batch_001.json", orders_batch_001)
list_landing()

### 4. Modo 1 - Full load didático
Lê todos os arquivos da pasta e sobrescreve a tabela. É simples, mas não é eficiente para grandes bases recorrentes.

In [0]:

full_df = (spark.read
    .format("json")
    .load(source_path)
    .withColumn("_source_file", col("_metadata.file_path"))
    .withColumn("_ingested_at", current_timestamp()))

(full_df.write
    .format("delta")
    .mode("overwrite")
    .option("overwriteSchema", "true")
    .saveAsTable(table_full))

spark.sql(f"SELECT COUNT(*) AS qtd_linhas FROM {table_full}").show()
display(spark.table(table_full).orderBy("pedido_id"))

### 5. Modo 2 - COPY INTO incremental e idempotente
Use este bloco para demonstrar carga incremental SQL-first e reexecução sem duplicar arquivos já processados.

In [0]:

spark.sql(f"""
    CREATE TABLE IF NOT EXISTS {table_copy} (
        pedido_id STRING,
        cliente_id STRING,
        produto_id STRING,
        quantidade BIGINT,
        valor_total DOUBLE,
        data_pedido STRING,
        status STRING,
        canal_venda STRING,
        _rescued_data STRING
    ) USING DELTA
""")

spark.sql(f"""
  COPY INTO {table_copy}
  FROM '{source_path}'
  FILEFORMAT = JSON
  FORMAT_OPTIONS ('multiLine' = 'false')
  COPY_OPTIONS ('mergeSchema' = 'true')
""")

spark.sql(f"SELECT COUNT(*) AS qtd_linhas FROM {table_copy}").show()
display(spark.table(table_copy).orderBy("pedido_id"))

In [0]:
# Reexecutar COPY INTO para demonstrar idempotência
spark.sql(f"""
    COPY INTO {table_copy}
    FROM '{source_path}'
    FILEFORMAT = JSON
    FORMAT_OPTIONS ('multiLine' = 'false')
    COPY_OPTIONS ('mergeSchema' = 'true')
""")

spark.sql(f"SELECT COUNT(*) AS qtd_linhas_apos_reexecucao FROM {table_copy}").show()

### 6. Gerar lote incremental
Este lote simula novos arquivos chegando à landing zone.

In [0]:

orders_batch_002 = [
    make_order(1005, "C004", "P100", 1, 29.95, 25, status="novo", channel="site"),
    make_order(1006, "C005", "P500", 4, 219.60, 31, status="novo", channel="site"),
    make_order(999, "C007", "P700", -5, 19.00, 23, status="novo", channel="whatsapp"),
]

write_jsonl(f"{source_path}/batch_002_incremental.json", orders_batch_002)
list_landing()

In [0]:

# Executar COPY INTO novamente para carregar apenas o novo arquivo
spark.sql(f"""
    COPY INTO {table_copy}
    FROM '{source_path}'
    FILEFORMAT = JSON
    FORMAT_OPTIONS ('multiLine' = 'false')
    COPY_OPTIONS ('mergeSchema' = 'true')
""")

spark.sql(f"SELECT COUNT(*) AS qtd_linhas FROM {table_copy}").show()
display(spark.table(table_copy).orderBy("pedido_id"))
     

### 7. Modo 3 - Auto Loader com availableNow
Este modo usa cloudFiles, checkpoint e schema location para controlar estado da ingestão.

In [0]:

# cloudFiles.schemaHints corrige os TIPOS (sem isso o Auto Loader infere tudo como STRING).
# O Auto Loader sempre ordena as colunas alfabeticamente ao inferir o schema.
# Para controlar a ORDEM, adicionamos um .select() explicito apos o .load().
schema_hints = (
    "pedido_id STRING, cliente_id STRING, produto_id STRING, "
    "quantidade BIGINT, valor_total DOUBLE, data_pedido STRING, "
    "status STRING, canal_venda STRING"
)

# Ordem desejada das colunas (igual ao COPY INTO + colunas de metadata)
col_order = [
    "pedido_id", "cliente_id", "produto_id",
    "quantidade", "valor_total", "data_pedido",
    "status", "canal_venda", "_rescued_data",
    "_source_file", "_ingested_at"
]

query = (spark.readStream
            .format("cloudFiles")
            .option("cloudFiles.format", "json")
            .option("cloudFiles.schemaLocation", schema_path)
            .option("cloudFiles.schemaEvolutionMode", "addNewColumns")
            .option("cloudFiles.schemaHints", schema_hints)
            .load(source_path)
            .withColumn("_source_file", col("_metadata.file_path"))
            .withColumn("_ingested_at", current_timestamp())
            .select(*col_order)  # força a ordem das colunas
            .writeStream
            .option("checkpointLocation", checkpoint_path)
            .option("mergeSchema", "true")
            .trigger(availableNow=True)
            .toTable(table_auto)
)
query.awaitTermination()

spark.sql(f"SELECT COUNT(*) AS qtd_linhas FROM {table_auto}").show()
display(spark.table(table_auto).orderBy("pedido_id"))
     

### 8. Teste de schema drift
O próximo arquivo adiciona cupom_desconto. Use para discutir schema evolution.

In [0]:
orders_batch_003_schema = [
    {**make_order(1007, "C006", "P600", 2, 179.80, 42, channel="app", status="novo"), "cupom_desconto": "APP10"},
    {**make_order(1008, "C007", "P700", 1, 299.90, 45, channel="web", status="novo"), "cupom_desconto": "WEB20"},
]

write_jsonl(f"{source_path}/batch_003_schema_drift.json", orders_batch_003_schema)
list_landing()
     

In [0]:
# Reexecutar Auto Loader para processar novo arquivo e evoluir schema
# 1º execução vai dar error - analisar!
# 2º execução com sucesso
query = (spark.readStream
    .format("cloudFiles")
    .option("cloudFiles.format", "json")
    .option("cloudFiles.schemaLocation", schema_path)
    .option("cloudFiles.schemaEvolutionMode", "addNewColumns")
    .load(source_path)
    .withColumn("_source_file", col("_metadata.file_path"))
    .withColumn("_ingested_at", current_timestamp())
    .writeStream
    .option("checkpointLocation", checkpoint_path)
    .option("mergeSchema", "true")
    .trigger(availableNow=True)
    .toTable(table_auto))

query.awaitTermination()
spark.sql(f"DESCRIBE TABLE {table_auto}").show(truncate=False)
display(spark.table(table_auto).orderBy("pedido_id"))

### 9. Modo 4 - CDC didático: preservar operações em Bronze
Não aplique MERGE ainda. A proposta do Dia 4 é preservar eventos; as regras podem ser tratadas no Dia 5.

In [0]:
cdc_path = f"{source_path}_cdc"
dbutils.fs.mkdirs(cdc_path)

cdc_events = [
    {"pedido_id":"2001","cliente_id":"C010","produto_id":"P900","quantidade":1,"valor_total":49.90,"data_pedido":"2026-07-27T12:00:00","op":"I","op_ts":"2026-07-27T12:01:00"},
    {"pedido_id":"2001","cliente_id":"C010","produto_id":"P900","quantidade":2,"valor_total":99.80,"data_pedido":"2026-07-27T12:00:00","op":"U","op_ts":"2026-07-27T12:05:00"},
    {"pedido_id":"2002","cliente_id":"C011","produto_id":"P901","quantidade":1,"valor_total":19.90,"data_pedido":"2026-07-27T12:02:00","op":"I","op_ts":"2026-07-27T12:03:00"},
    {"pedido_id":"2002","cliente_id":"C011","produto_id":"P901","quantidade":1,"valor_total":19.90,"data_pedido":"2026-07-27T12:02:00","op":"D","op_ts":"2026-07-27T12:08:00"},
]

write_jsonl(f"{cdc_path}/cdc_batch_001.json", cdc_events)

cdc_df = (spark.read
    .format("json")
    .load(cdc_path)
    .withColumn("_source_file", col("_metadata.file_path"))
    .withColumn("_ingested_at", current_timestamp())
)

(cdc_df.write
    .format("delta")
    .mode("overwrite")
    .option("overwriteSchema", "true")
    .saveAsTable(table_cdc_raw))

spark.sql(f"SELECT op, COUNT(*) AS qtd FROM {table_cdc_raw} GROUP BY op ORDER BY op").show()
display(spark.table(table_cdc_raw).orderBy("pedido_id", "op_ts"))
     

### 10. Validações finais
Use estas consultas como evidências objetivas do laboratório.

In [0]:
validation_queries = [
    f"SELECT 'full_load' AS tabela, COUNT(*) AS linhas FROM {table_full}",
    f"SELECT 'copy_into' AS tabela, COUNT(*) AS linhas FROM {table_copy}",
    f"SELECT 'autoloader' AS tabela, COUNT(*) AS linhas FROM {table_auto}",
    f"SELECT 'cdc_raw' AS tabela, COUNT(*) AS linhas FROM {table_cdc_raw}",
]

for q in validation_queries:
    spark.sql(q).show()

spark.sql(f"DESCRIBE TABLE EXTENDED {table_auto}").show(truncate=False)

In [0]:

# Execute apenas se quiser reiniciar o exercício do zero
RESET_LAB = False

if RESET_LAB:
    for t in [table_full, table_copy, table_auto, table_cdc_raw]:
        spark.sql(f"DROP TABLE IF EXISTS {t}")
        
    dbutils.fs.rm(source_path, recurse=True)
    dbutils.fs.rm(checkpoint_path, recurse=True)
    dbutils.fs.rm(schema_path, recurse=True)
    dbutils.fs.rm(cdc_path, recurse=True)
    print("Laboratório reiniciado.")
else:
    print("RESET_LAB = False. Nenhum objeto foi removido.")

### Perguntas de fixação

1. Quando o full load é aceitável?
2. Por que COPY INTO não duplicou os arquivos já carregados?
3. Qual é o papel do checkpoint no Auto Loader?
4. Por que schema location não deve ser tratado como pasta temporária?
5. Por que a Bronze deve preservar eventos CDC sem aplicar regra de negócio imediatamente?